In [ ]:
import pandas as pd
import numpy as np
import csv
import seaborn as sns
import matplotlib.pyplot as plt

##import data files here

# Chi squared 
to identify which regions are significantly different 

In [ ]:
from scipy.stats import chi2_contingency

results = []

for idx, row in final_df.iterrows():
    table = [
        [row['SK_with'], row['SK_without']],
        [row['CMH_with'], row['CMH_without']],
        [row['NCH_with'], row['NCH_without']]
    ]
    
    if sum([sum(x) for x in table]) < 5 or any(sum(x) == 0 for x in table):
        continue

    chi2, p, dof, expected = chi2_contingency(table)
    results.append({
        'Lesion_location_region': row['Lesion_location_region'],
        'chi2': chi2,
        'p_value': p
    })

# Turn into DataFrame
chi2_df = pd.DataFrame(results)

significant = chi2_df[chi2_df['p_value'] < 0.05]

print(significant.sort_values(by='p_value'))

### post hocs and correcting for multiple comparisons 

In [ ]:
from scipy.stats import chi2_contingency
import itertools

# Function to run chi-square on 2x2 contingency tables
def pairwise_chi2(name, row, total_patients):
    pairs = [('SK', 'CMH'), ('SK', 'NCH'), ('CMH', 'NCH')]
    results = []

    for a, b in pairs:
        table = [
            [row[f'{a}_with'], row[f'{a}_without']],
            [row[f'{b}_with'], row[f'{b}_without']]
        ]
        
        table = np.array(table)

        if (table.sum(axis=0) == 0).any() or (table.sum(axis=1) == 0).any():
            continue

        try:
            chi2, p, _, _ = chi2_contingency(table)
            results.append({
                'Lesion_location_region': name,
                'Comparison': f'{a} vs {b}',
                'p_value': p
            })
        except ValueError:
            continue

    return results

all_results = []

for idx, row in final_df.iterrows():
    # Overall 3x2 test
    table = [
        [row['SK_with'], row['SK_without']],
        [row['CMH_with'], row['CMH_without']],
        [row['NCH_with'], row['NCH_without']]
    ]

    if sum([sum(x) for x in table]) < 5 or any(sum(x) == 0 for x in table):
        continue

    chi2, p, dof, expected = chi2_contingency(table)

    if p < 0.05:  
        pairwise_results = pairwise_chi2(row['Lesion_location_region'], row, total_patients)
        
        # Apply Bonferroni correction
        for res in pairwise_results:
            res['p_corrected'] = min(res['p_value'] * 3, 1.0)  # 3 comparisons
            all_results.append(res)

posthoc_df = pd.DataFrame(all_results)
significant_pairs = posthoc_df[posthoc_df['p_corrected'] < 0.05]

print(significant_pairs.sort_values(by='p_corrected'))

## Fischer exact with monte carlo 

In [ ]:

from scipy.stats import fisher_exact, MonteCarloMethod

results = []


mc_method = MonteCarloMethod(n_resamples=10000, rng=np.random.default_rng(42))

for idx, row in final_df.iterrows():
    table = [
        [row['SK_with'], row['SK_without']],
        [row['CMH_with'], row['CMH_without']],
        [row['NCH_with'], row['NCH_without']]
    ]

    
    if sum(sum(r) for r in table) < 5 or any(sum(r) == 0 for r in table):
        continue

    
    result = fisher_exact(table, method=mc_method)
    p_value = result.pvalue  # Monte Carlo estimated p‑value
    
    results.append({
        'Lesion_location_region': row['Lesion_location_region'],
        'p_value': p_value
    })


fisher_df = pd.DataFrame(results)


significant = fisher_df[fisher_df['p_value'] < 0.05]
print(significant.sort_values(by='p_value'))


In [ ]:

def pairwise_fisher(name, row):
    pairs = [('SK', 'CMH'), ('SK', 'NCH'), ('CMH', 'NCH')]
    results = []

    for a, b in pairs:
        table = [
            [row[f'{a}_with'], row[f'{a}_without']],
            [row[f'{b}_with'], row[f'{b}_without']]
        ]
        
        table = np.array(table)
        if (table.sum(axis=0) == 0).any() or (table.sum(axis=1) == 0).any():
            continue
        
        _, p = fisher_exact(table)
        
        results.append({
            'Lesion_location_region': name,
            'Comparison': f'{a} vs {b}',
            'p_value': p,
            'p_corrected': min(p * 3, 1.0)
        })
    
    return results

# Store post-hoc results
all_posthoc_results = []

for idx, row in final_df.iterrows():
    
    table = [
        [row['SK_with'], row['SK_without']],
        [row['CMH_with'], row['CMH_without']],
        [row['NCH_with'], row['NCH_without']]
    ]
    
    if sum(sum(r) for r in table) < 5 or any(sum(r) == 0 for r in table):
        continue

    from scipy.stats import MonteCarloMethod, fisher_exact
    mc_method = MonteCarloMethod(n_resamples=10000, rng=np.random.default_rng(42))
    overall_result = fisher_exact(table, method=mc_method)
    
    if overall_result.pvalue < 0.05:
        pairwise_results = pairwise_fisher(row['Lesion_location_region'], row)
        all_posthoc_results.extend(pairwise_results)

posthoc_df = pd.DataFrame(all_posthoc_results)
significant_pairs = posthoc_df[posthoc_df['p_corrected'] < 0.05]

print(significant_pairs.sort_values(by='p_corrected'))


# comparing lesion co-occurances
seing if the patterns are different between hospitals

In [ ]:
## import data files here 

In [ ]:
pip install scikit-bio


In [ ]:
from itertools import combinations_with_replacement
from scipy.spatial.distance import pdist, squareform

def build_cooccurrence_matrix(df):
    lesions = sorted(df['Lesion_location_region'].dropna().unique())
    cooc_mat = pd.DataFrame(0, index=lesions, columns=lesions, dtype=int)

    grouped = df.groupby('record_id')['Lesion_location_region'].apply(lambda x: x.dropna().tolist())

    for lesions_list in grouped:
        lesion_counts = pd.Series(lesions_list).value_counts()

        for lesion1, lesion2 in combinations_with_replacement(lesion_counts.index, 2):
            count1 = lesion_counts[lesion1]
            count2 = lesion_counts[lesion2]
            if lesion1 == lesion2:
                pair_count = count1 * (count1 - 1) // 2
            else:
                pair_count = count1 * count2
            cooc_mat.loc[lesion1, lesion2] += pair_count
            if lesion1 != lesion2:
                cooc_mat.loc[lesion2, lesion1] += pair_count

    return cooc_mat

site_matrices = []
site_names = []

for site_name, df_site in [('SK', SKoveral_lesions), ('CMH', CMHoveral_lesions), ('NCH', NCHoveral_lesions)]:
    mat = build_cooccurrence_matrix(df_site)
    site_matrices.append(mat)
    site_names.append(site_name)

# 1. Get the union of all lesion locations (rows and columns)
all_lesions = sorted(set().union(*[set(mat.index) for mat in site_matrices]))

# 2. Reindex each matrix to all lesions, filling missing with 0
aligned_matrices = []
for mat in site_matrices:
    mat_aligned = mat.reindex(index=all_lesions, columns=all_lesions, fill_value=0)
    aligned_matrices.append(mat_aligned)

# Get upper triangle indices (exclude diagonal)
triu_idx = np.triu_indices(len(all_lesions), k=1)

# Flatten only unordered pairs
vectors = [mat.values[triu_idx] for mat in aligned_matrices]

# Stack into 2D array
vectors = np.vstack(vectors)

# 5. Calculate pairwise Euclidean distance between site vectors
dist_mat = squareform(pdist(vectors, metric='euclidean'))

# 6. Output
print("Sites:", site_names)
print("Distance matrix between sites:\n", dist_mat)


In [ ]:
from itertools import combinations_with_replacement
from skbio.stats.distance import DistanceMatrix, permanova
from scipy.spatial.distance import pdist, squareform

def build_patient_cooc_vector(patient_lesions, global_lesions):
    lesion_counts = pd.Series(patient_lesions).value_counts()
    cooc = pd.DataFrame(0, index=global_lesions, columns=global_lesions, dtype=int)
    
    for lesion1, lesion2 in combinations_with_replacement(lesion_counts.index, 2):
        count1 = lesion_counts[lesion1]
        count2 = lesion_counts[lesion2]
        if lesion1 == lesion2:
            pair_count = count1 * (count1 - 1) // 2
        else:
            pair_count = count1 * count2
        cooc.loc[lesion1, lesion2] += pair_count
        if lesion1 != lesion2:
            cooc.loc[lesion2, lesion1] += pair_count

    vec = cooc.values[np.triu_indices(len(global_lesions), k=1)]

    return vec


# 1) Find the global set of lesions (union across all sites)
all_lesions = set()
for df in site_dfs.values():
    all_lesions.update(df['Lesion_location_region'].dropna().unique())
global_lesions = sorted(all_lesions)

all_vectors = []
all_labels = []
all_patient_ids = []

# 2) For each site, compute patient vectors with global lesion list
for site_name, df_site in site_dfs.items():
    grouped = df_site.groupby('record_id')['Lesion_location_region'].apply(lambda x: x.dropna().tolist())
    
    for patient_id, lesions_list in grouped.items():
        vec = build_patient_cooc_vector(lesions_list, global_lesions)
        all_vectors.append(vec)
        all_labels.append(site_name)
        all_patient_ids.append(patient_id)

all_vectors = np.vstack(all_vectors)

# 3) Compute patient-patient distances (Euclidean)
dist_mat = squareform(pdist(all_vectors, metric='euclidean'))

# 4) Create DistanceMatrix and metadata
dm = DistanceMatrix(dist_mat, ids=all_patient_ids)
metadata = pd.DataFrame({'site': all_labels}, index=all_patient_ids)

# 5) Run PERMANOVA
results = permanova(dm, metadata, column='site', permutations=999)

print(results)


In [ ]:
from itertools import combinations
from statsmodels.stats.multitest import multipletests


sites = metadata['site'].unique()
pairwise_results = []

for site1, site2 in combinations(sites, 2):
    ids_subset = metadata.index[metadata['site'].isin([site1, site2])]
    dm_subset = dm.filter(ids_subset)
    meta_subset = metadata.loc[ids_subset]

    result = permanova(dm_subset, meta_subset, column='site', permutations=999)
    pairwise_results.append({
        'site1': site1,
        'site2': site2,
        'pseudo-F': result['test statistic'],
        'p-value': result['p-value']
    })

pairwise_df = pd.DataFrame(pairwise_results)

pvals = pairwise_df['p-value']
pairwise_df['p_fdr'] = multipletests(pvals, method='fdr_bh')[1]


print(pairwise_df)


In [ ]:
from collections import Counter

# Initialize dictionary to hold summed co-occurrence matrices for each site
sum_cooc_matrices = {
    site: np.zeros((len(global_lesions), len(global_lesions)), dtype=int)
    for site in site_dfs
}

for site_name, df_site in site_dfs.items():
    grouped = df_site.groupby('record_id')['Lesion_location_region'] \
                     .apply(lambda x: x.dropna().tolist())

    for patient_id, lesions_list in grouped.items():
        lesion_indices = [global_lesions.index(l) for l in lesions_list if l in global_lesions]
        cooc_matrix = np.zeros((len(global_lesions), len(global_lesions)), dtype=int)

        counts = Counter(lesion_indices)
        for lesion_idx, count in counts.items():
            if count >= 2:
                cooc_matrix[lesion_idx, lesion_idx] += count * (count - 1) // 2

        for i, j in combinations(lesion_indices, 2):
            cooc_matrix[i, j] += 1
            cooc_matrix[j, i] += 1  # keep symmetric

        sum_cooc_matrices[site_name] += cooc_matrix


# Find top 3 most common lesion pairs per site (excluding diagonals)
for site, matrix in sum_cooc_matrices.items():
    # Ignore diagonal (same lesion pairs)
    np.fill_diagonal(matrix, 0)

    # Get top 3 unordered pairs (upper triangle only)
    triu_indices = np.triu_indices_from(matrix, k=1)
    values = matrix[triu_indices]

    top_idx = np.argpartition(values, -3)[-3:]
    top_pairs = sorted(top_idx, key=lambda x: values[x], reverse=True)

    print(f"Top 3 lesion co-occurrence pairs for site {site}:")
    for rank, idx in enumerate(top_pairs, 1):
        i = triu_indices[0][idx]
        j = triu_indices[1][idx]
        print(f"{rank}. ({global_lesions[i]}, {global_lesions[j]}) - Count: {matrix[i, j]}")
    print()
